# 1D Cutting Stock Problem — Formulation and Solution (Python)

> **ORKit Free** · Mixed-Integer Programming (MIP)

---

## What is the Cutting Stock Problem?

Imagine a paper mill that produces large master rolls of width $W$. Customers order smaller pieces
of different widths. The goal is to **cut the master rolls into the requested pieces using as few
rolls as possible**, minimizing waste.

This classical problem was first studied by **Kantorovich (1960)** and is foundational in:
- Paper and steel industries
- Textile cutting
- Glass manufacturing
- Any process where large raw materials must be cut into smaller pieces

---

## Mathematical Formulation

### Sets and Parameters

| Symbol | Description |
|--------|-------------|
| $I$ | Set of item types |
| $N$ | Set of available rolls (upper bound: $\sum_i d_i$) |
| $W$ | Width of the master roll |
| $w_i$ | Width of item type $i$ |
| $d_i$ | Demand for item type $i$ |

### Decision Variables

- $y_n \in \{0, 1\}$: 1 if roll $n$ is opened (used)
- $x_{in} \in \mathbb{Z}_+$: number of pieces of type $i$ cut from roll $n$

### Formulation

$$\min \quad \sum_{n \in N} y_n$$

$$\text{s.t.} \quad \sum_{n \in N} x_{in} \geq d_i, \quad \forall i \in I \quad \text{(demand)}$$

$$\sum_{i \in I} w_i \, x_{in} \leq W \, y_n, \quad \forall n \in N \quad \text{(capacity)}$$

$$y_n \in \{0,1\}, \quad x_{in} \in \mathbb{Z}_+$$

**Interpretation:**
- We minimize the number of master rolls used
- Each item type demand must be satisfied
- The total width of items cut from each roll cannot exceed the master roll width
- Items can only be cut from an open roll ($x_{in} > 0 \Rightarrow y_n = 1$)

## Setting Up the Environment

We use **Pyomo** as the modeling language and **HiGHS** as the open-source MIP solver.

In [ ]:
import pyomo.environ as pyo
import json

print(f"Pyomo version: {pyo.__version__}")

## Loading the Instance

Our small test instance has:
- Master roll width: **100 units**
- 3 item types with different widths and demands

In [ ]:
# Load instance from JSON
with open("../instances/small_3.json") as f:
    data = json.load(f)

W = data["master_roll"]
items = data["items"]

print(f"Master roll width: {W}")
print(f"Number of item types: {len(items)}")
print()
for it in items:
    print(f"  Item {it['id']}: width={it['width']}, demand={it['demand']}")

## Building the Pyomo Model — Step by Step

### Step 1: Compute the upper bound on rolls

A trivial upper bound: in the worst case we use one roll per demanded piece.
This gives us $N_{\max} = \sum_i d_i$.

In [ ]:
n_max = sum(it["demand"] for it in items)
print(f"Upper bound on rolls (N_max): {n_max}")
print(f"This means we need at MOST {n_max} rolls.")
print(f"The solver will find how many we ACTUALLY need.")

### Step 2: Create the model with sets, parameters, and variables

In [ ]:
model = pyo.ConcreteModel(name="CuttingStock_1D")

# ── Sets ──────────────────────────────────────────
item_ids = [it["id"] for it in items]
model.I = pyo.Set(initialize=item_ids, doc="Item types")
model.N = pyo.RangeSet(1, n_max, doc="Available rolls")

# ── Parameters ────────────────────────────────────
model.w = pyo.Param(model.I, initialize={it["id"]: it["width"] for it in items})
model.d = pyo.Param(model.I, initialize={it["id"]: it["demand"] for it in items})
model.W = pyo.Param(initialize=W)

# ── Decision Variables ────────────────────────────
model.y = pyo.Var(model.N, within=pyo.Binary, doc="1 if roll n is opened")
model.x = pyo.Var(model.I, model.N, within=pyo.NonNegativeIntegers,
                   doc="pieces of type i cut from roll n")

print(f"Variables y: {len(list(model.y))} (one per roll)")
print(f"Variables x: {len(list(model.x))} (items x rolls)")

### Step 3: Define the objective and constraints

In [ ]:
# ── Objective: minimize rolls used ────────────────
model.obj = pyo.Objective(
    expr=sum(model.y[n] for n in model.N),
    sense=pyo.minimize,
)

# ── Demand constraints ────────────────────────────
# Each item type demand must be fully met
def demand_rule(m, i):
    return sum(m.x[i, n] for n in m.N) >= m.d[i]

model.demand = pyo.Constraint(model.I, rule=demand_rule)

# ── Capacity constraints ──────────────────────────
# Total width of items on each roll <= master roll width (only if opened)
def capacity_rule(m, n):
    return sum(m.w[i] * m.x[i, n] for i in m.I) <= m.W * m.y[n]

model.capacity = pyo.Constraint(model.N, rule=capacity_rule)

print("Model built successfully!")
print(f"  Constraints (demand): {len(list(model.demand))}")
print(f"  Constraints (capacity): {len(list(model.capacity))}")

## Solving the Model

In [ ]:
solver = pyo.SolverFactory("appsi_highs")
result = solver.solve(model, tee=False)

print(f"Solver status: {result.solver.termination_condition}")
print(f"Rolls used: {int(round(pyo.value(model.obj)))}")

## Analyzing the Solution

Let's see which rolls were opened and what cutting patterns were used.

In [ ]:
print("Cutting Patterns")
print("=" * 50)

total_waste = 0
for n in model.N:
    if pyo.value(model.y[n]) > 0.5:
        cuts = {}
        used_width = 0
        for i in model.I:
            qty = int(round(pyo.value(model.x[i, n])))
            if qty > 0:
                cuts[i] = qty
                used_width += items[i - 1]["width"] * qty
        waste = W - used_width
        total_waste += waste
        print(f"  Roll {n}: {cuts}  (used={used_width}, waste={waste})")

rolls_used = int(round(pyo.value(model.obj)))
print(f"\nTotal rolls: {rolls_used}")
print(f"Total waste: {total_waste} ({total_waste / (rolls_used * W) * 100:.1f}%)")

## Demand Verification

Let's confirm that every item type demand was satisfied.

In [ ]:
print(f"{'Item':>6} {'Width':>6} {'Demand':>7} {'Cut':>5} {'OK?':>5}")
print("-" * 35)
for it in items:
    i = it["id"]
    total_cut = sum(int(round(pyo.value(model.x[i, n]))) for n in model.N)
    ok = "YES" if total_cut >= it["demand"] else "NO"
    print(f"{i:>6} {it['width']:>6} {it['demand']:>7} {total_cut:>5} {ok:>5}")

## Key Takeaways

1. **The compact MIP formulation** uses binary variables $y_n$ and integer variables $x_{in}$.
   It is straightforward but grows quickly with the number of possible rolls.

2. **For large instances**, the Dantzig-Wolfe decomposition (column generation) is preferred:
   instead of enumerating all possible rolls, we generate cutting patterns on-the-fly.
   This is covered in the **Master** bundle.

3. **Waste minimization** is equivalent to minimizing rolls: fewer rolls = less raw material = less waste.

---

### References

- Kantorovich, L. V. (1960). Mathematical Methods of Organising and Planning Production.
- Wascher, G., Haussner, H., Schumann, H. (2007). An improved typology of cutting and packing problems. *EJOR*, 183(3), 1109-1130.
- Gilmore, P. C., Gomory, R. E. (1961). A linear programming approach to the cutting stock problem. *Operations Research*, 9(6), 849-859.